# Notebook to test installation, MiRepNet utilities, correct weights, cuda, etc...

In [1]:
# Test installation and imports
import torch
import torchvision
import torchaudio
import numpy as np
import scipy
import mne
import pandas as pd
import sklearn
import matplotlib
import pylsl
import einops
import wandb
import moabb
import tqdm

print("All libraries imported successfully.")

# Check PyTorch device
if torch.cuda.is_available():
    print(f"PyTorch device: cuda ({torch.cuda.get_device_name(0)})")
else:
    print("PyTorch device: cpu")

c:\Users\mathi\anaconda3\envs\hacktion_potential\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All libraries imported successfully.
PyTorch device: cuda (NVIDIA GeForce RTX 4050 Laptop GPU)


In [3]:
# Import MiRepNet and sanity check model
import torch
from model.mlm import mlm_mask

# Instantiate model (adjust emb_size, depth, n_classes as needed)
model = mlm_mask(emb_size=256, depth=6, n_classes=3)  # match checkpoint params

# Load pretrained weights if available
pretrained_path = './weight/MIRepNet.pth'
try:
    state_dict = torch.load(pretrained_path, map_location=torch.device('cpu'))
    model.load_state_dict(state_dict, strict=False)  # allow missing/unexpected keys
    print('Pretrained weights loaded to CPU (non-strict).')
except Exception as e:
    print('Could not load pretrained weights:', e)

# Sanity check: forward pass with dummy data
# shape: (batch, channels, samples)
dummy_input = torch.randn(1, 45, 3000)
with torch.no_grad():
    pooled, cls_output = model(dummy_input)
print('Sanity check output:', cls_output)

Pretrained weights loaded to CPU (non-strict).
Sanity check output: tensor([[ 3.6084, -0.9981,  0.0235]])


C:\Users\mathi\AppData\Local\Temp\ipykernel_23800\1603936767.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(pretrained_path, map_location=torch

In [ ]:
# print information of the model
print(model)

mlm_mask(
  (embedding): PatchEmbedding(
    (conv1): Conv2d(1, 64, kernel_size=(1, 25), stride=(1, 1))
    (conv2): Conv2d(64, 128, kernel_size=(45, 1), stride=(1, 1))
    (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (elu): ELU(alpha=1.0)
    (pool): AvgPool2d(kernel_size=(1, 75), stride=(1, 15), padding=0)
    (dropout): Dropout(p=0.5, inplace=False)
    (projection): Sequential(
      (0): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1))
      (1): Rearrange('b e (h) (w) -> b (h w) e')
    )
    (chan_embed): Embedding(45, 256)
  )
  (transformer): TransformerEncoder(
    (0): TransformerEncoderBlock(
      (0): ResidualAdd(
        (fn): Sequential(
          (0): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (1): MultiHeadAttention(
            (keys): Linear(in_features=256, out_features=256, bias=True)
            (queries): Linear(in_features=256, out_features=256, bias=True)
            (values): Linear(in_feat